In [82]:
import pandas as pd

In [83]:
projects = pd.read_csv("../data/projects.csv")
milestones = pd.read_csv("../data/milestones.csv")
costs = pd.read_csv("../data/costs.csv")

In [84]:
milestones["planned_date"] = pd.to_datetime(
    milestones["planned_date"],
    errors="coerce"
)

milestones["actual_date"] = pd.to_datetime(
    milestones["actual_date"],
    errors="coerce"
)

In [85]:
projects_by_status = (projects.groupby("status").size().sort_values(ascending=False))

projects_by_status

status
In Progress    13
Done            3
Blocked         2
Planned         2
dtype: int64

In [86]:
project_costs = (costs.groupby("project_id")["amount"].sum())

In [87]:
project_costs = project_costs.reset_index()

In [88]:
project_costs.head()

,project_id,amount
0,1,53500.0
1,2,39200.0
2,3,81200.0
3,4,56600.0
4,5,32600.0


In [89]:
project_costs = project_costs.rename(columns={"amount": "total_cost"})

In [90]:
project_costs = project_costs.rename(columns={"project_id": "id"})

In [91]:
project_metrics = projects.merge(project_costs, on="id", how="left", validate="one_to_one")

In [92]:
project_metrics.head()

,id,customer_id,name,start_date,target_go_live_date,status,budget_amount,total_cost
0,1,1,Core Rollout,2025-12-18,2026-02-10,In Progress,60000.0,53500.0
1,2,1,Portal Launch,2026-01-04,2026-03-01,In Progress,45000.0,39200.0
2,3,2,POS Upgrade,2025-11-25,2026-01-20,Done,90000.0,81200.0
3,4,2,Loyalty Setup,2026-01-06,2026-03-10,In Progress,70000.0,56600.0
4,5,3,Supplier Hub,2025-12-30,2026-02-20,In Progress,33000.0,32600.0


In [93]:
project_metrics["budget_utilization_pct"] = (project_metrics["total_cost"] / project_metrics["budget_amount"] * 100).round(1)

In [94]:
project_metrics.head()

,id,customer_id,name,start_date,target_go_live_date,status,budget_amount,total_cost,budget_utilization_pct
0,1,1,Core Rollout,2025-12-18,2026-02-10,In Progress,60000.0,53500.0,89.2
1,2,1,Portal Launch,2026-01-04,2026-03-01,In Progress,45000.0,39200.0,87.1
2,3,2,POS Upgrade,2025-11-25,2026-01-20,Done,90000.0,81200.0,90.2
3,4,2,Loyalty Setup,2026-01-06,2026-03-10,In Progress,70000.0,56600.0,80.9
4,5,3,Supplier Hub,2025-12-30,2026-02-20,In Progress,33000.0,32600.0,98.8


In [95]:
status_date = pd.Timestamp("2026-09-28")

In [96]:
overdue_mask = ((milestones["planned_date"] < status_date) & (milestones["actual_date"].isna()))

In [97]:
overdue_milestones = milestones[overdue_mask]

In [98]:
overdue_milestones.head()

,id,project_id,type,planned_date,actual_date,status
1,2,1,Discovery,2026-01-05,NaT,In Progress
2,3,1,Build,2026-01-20,NaT,Planned
4,5,2,Discovery,2026-01-12,NaT,In Progress
5,6,2,Integration,2026-01-26,NaT,Planned
10,11,4,Discovery,2026-01-14,NaT,In Progress


In [99]:
overdue_by_project = (overdue_milestones.groupby("project_id").size().reset_index(name="overdue_milestones"))

In [100]:
overdue_by_project = overdue_by_project.rename(columns={"project_id": "id"})

In [101]:
overdue_by_project.head()

,id,overdue_milestones
0,1,2
1,2,2
2,4,2
3,5,2
4,6,3


In [102]:
project_metrics = project_metrics.merge(overdue_by_project, on="id", how="left", validate="one_to_one")

In [103]:
project_metrics.head()

,id,customer_id,name,start_date,target_go_live_date,status,budget_amount,total_cost,budget_utilization_pct,overdue_milestones
0,1,1,Core Rollout,2025-12-18,2026-02-10,In Progress,60000.0,53500.0,89.2,2.0
1,2,1,Portal Launch,2026-01-04,2026-03-01,In Progress,45000.0,39200.0,87.1,2.0
2,3,2,POS Upgrade,2025-11-25,2026-01-20,Done,90000.0,81200.0,90.2,NaN
3,4,2,Loyalty Setup,2026-01-06,2026-03-10,In Progress,70000.0,56600.0,80.9,2.0
4,5,3,Supplier Hub,2025-12-30,2026-02-20,In Progress,33000.0,32600.0,98.8,2.0


In [107]:
project_metrics["overdue_milestones"] = (project_metrics["overdue_milestones"].fillna(0).astype(int))

In [108]:
project_metrics.head()

,id,customer_id,name,start_date,target_go_live_date,status,budget_amount,total_cost,budget_utilization_pct,overdue_milestones
0,1,1,Core Rollout,2025-12-18,2026-02-10,In Progress,60000.0,53500.0,89.2,2
1,2,1,Portal Launch,2026-01-04,2026-03-01,In Progress,45000.0,39200.0,87.1,2
2,3,2,POS Upgrade,2025-11-25,2026-01-20,Done,90000.0,81200.0,90.2,0
3,4,2,Loyalty Setup,2026-01-06,2026-03-10,In Progress,70000.0,56600.0,80.9,2
4,5,3,Supplier Hub,2025-12-30,2026-02-20,In Progress,33000.0,32600.0,98.8,2


In [109]:
project_metrics[["name", "status", "budget_amount", "total_cost", "budget_utilization_pct", "overdue_milestones"]]

,name,status,budget_amount,total_cost,budget_utilization_pct,overdue_milestones
0,Core Rollout,In Progress,60000.0,53500.0,89.2,2
1,Portal Launch,In Progress,45000.0,39200.0,87.1,2
2,POS Upgrade,Done,90000.0,81200.0,90.2,0
3,Loyalty Setup,In Progress,70000.0,56600.0,80.9,2
4,Supplier Hub,In Progress,33000.0,32600.0,98.8,2
5,Reporting Setup,Planned,25000.0,3200.0,12.8,3
6,Workflow Upgrade,In Progress,80000.0,98000.0,122.5,2
7,CRM Integration,In Progress,70000.0,48700.0,69.6,2
8,Scheduling Deployment,Blocked,50000.0,33200.0,66.4,2
9,Asset Dashboard,In Progress,41000.0,40100.0,97.8,2
